# Exercise 3: Neural networks in PyTorch

In this exercise you’ll implement small neural-network building blocks from scratch and use them to train a simple classifier.

You’ll cover:
- **Basic layers**: Linear, Embedding, Dropout
- **Normalization**: LayerNorm and RMSNorm
- **MLPs + residual**: composing layers into deeper networks
- **Classification**: generating a learnable dataset, implementing cross-entropy from logits, and writing a minimal training loop

As before: fill in all `TODO`s without changing function names or signatures.
Use small sanity checks and compare to PyTorch reference implementations when useful.

In [2]:
from __future__ import annotations

import torch
from torch import nn

## Basic layers

In this section you’ll implement a few core layers that appear everywhere:

### `Linear`
A fully-connected layer that follows nn.Linear conventions:  
`y = x @ Wᵀ + b`

Important details:
- Parameters should be registered as `nn.Parameter`
- Store weight as (out_features, in_features) like nn.Linear.
- The forward pass should support leading batch dimensions: `x` can be shape `(..., in_features)`

### `Embedding`
An embedding table maps integer ids to vectors:
- input: token ids `idx` of shape `(...,)`
- output: vectors of shape `(..., embedding_dim)`

This is essentially a learnable lookup table.

### `Dropout`
Dropout randomly zeroes activations during training to reduce overfitting.
Implementation details:
- Only active in `model.train()` mode
- In training: drop with probability `p` and scale the kept values by `1/(1-p)` so the expected value stays the same
- In eval: return the input unchanged

## Instructions
- Do not use PyTorch reference modules for the parts you implement (e.g. don’t call nn.Linear inside your Linear).
- You may use standard tensor ops that you learned before (matmul, sum, mean, rsqrt, indexing, etc.).
- Use a parameter initialization method of your choice. We recommend something like Xavier-uniform.


In [6]:
class Linear(nn.Module):
    def __init__(self, in_features: int, out_features: int, bias: bool = True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        self.weight = nn.Parameter(torch.empty(self.out_features, self.in_features))
        if bias:
            self.bias = nn.Parameter(torch.empty(out_features))
        else:
            self.bias = None 
        
        nn.init.xavier_uniform_(self.weight)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if self.bias is not None:
            y = (x @ self.weight.T) + self.bias
        else:
            y = (x @ self.weight.T)           
        return y
         
        

In [7]:
in_f, out_f = 3, 4
layer = Linear(in_f, out_f, bias=True)

x1 = torch.randn(2, in_f)       # (batch, in_features)
x2 = torch.randn(5, 2, in_f)    # (batch, seq, in_features)

print(layer(x1).shape)  # expect (2, out_f)
print(layer(x2).shape)  # expect (5, 2, out_f)


torch.Size([2, 4])
torch.Size([5, 2, 4])


In [9]:
class Embedding(nn.Module):
    def __init__(self, num_embeddings: int, embedding_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_embeddings, embedding_dim))
        nn.init.xavier_uniform_(self.weight)


    def forward(self, idx: torch.Tensor) -> torch.Tensor:
        """
        idx: (...,) int64
        return: (..., embedding_dim)
        """
        return self.weight[idx]


In [10]:
emb = Embedding(num_embeddings=5, embedding_dim=4)

# Case 1: batch of token IDs: (batch,)
idx = torch.tensor([0, 3, 1], dtype=torch.long)  # shape (3,)
out = emb(idx)
print("idx shape:", idx.shape)
print("out shape:", out.shape)   # (3, 4)

# Case 2: batch of sequences: (batch, seq_len)
idx_seq = torch.tensor([[0, 1, 2],
                        [3, 4, 0]], dtype=torch.long)  # shape (2, 3)
out_seq = emb(idx_seq)
print("idx_seq shape:", idx_seq.shape)
print("out_seq shape:", out_seq.shape)  # (2, 3, 4)

idx shape: torch.Size([3])
out shape: torch.Size([3, 4])
idx_seq shape: torch.Size([2, 3])
out_seq shape: torch.Size([2, 3, 4])


In [ ]:
class Dropout(nn.Module):
    def __init__(self, p: float):
        super().__init__()
        self.p = p

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        In train mode: drop with prob p and scale by 1/(1-p).
        In eval mode: return x unchanged.
        """
        if self.training:
            m = torch.bernoulli(torch.full_like(x, 1 - self.p))
            y = x * m / (1 - self.p)
            return y
        else:
            return x

## Normalization

Normalization layers help stabilize training by controlling activation statistics.

### LayerNorm
LayerNorm normalizes each example across its **feature dimension** (the last dimension):

- compute mean and variance over the last dimension
- normalize: `(x - mean) / sqrt(var + eps)`
- apply learnable per-feature scale and shift (`weight`, `bias`)

**In this exercise, assume `elementwise_affine=True` (always include `weight` and `bias`).**  
`weight` and `bias` each have shape `(D,)`.

LayerNorm is widely used in transformers because it does not depend on batch statistics.

### RMSNorm
RMSNorm is similar to LayerNorm but normalizes using only the root-mean-square:
- `x / sqrt(mean(x^2) + eps)` over the last dimension
- usually includes a learnable scale (`weight`)
- no mean subtraction

RMSNorm is popular in modern LLMs because it's faster.


In [ ]:
class LayerNorm(nn.Module):
    def __init__(
        self, normalized_shape: int, eps: float = 1e-5, elementwise_affine: bool = True
    ):
        super().__init__()
        self.eps = eps
        self.elementwise_affine = elementwise_affine
        self.normalized_shape = normalized_shape

        if self.elementwise_affine:
            self.weight = nn.Parameter(torch.ones(normalized_shape))
            self.bias = nn.Parameter(torch.ones(normalized_shape))
        else:
            self.register_parameter("weight", None)
            self.register_parameter("bias", None)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Normalize over the last dimension.
        x: (..., D)
        """
        mean = x.mean(dim=-1, keepdim=True)
        variance = x.var(dim=-1, keepdim=True, unbiased=False)
        
        x_hat = (x-mean)/torch.sqrt(variance+self.eps)

        if self.elementwise_affine:
            return x_hat * self.weight + self.bias
        else:
            return x_hat

In [20]:
torch.manual_seed(0)
x = torch.randn(2, 3, 4)  # (..., D) with D=4

my = LayerNorm(4)
ref = nn.LayerNorm(4, elementwise_affine=True)

with torch.no_grad():
    ref.weight.copy_(my.weight)
    ref.bias.copy_(my.bias)

y_my = my(x)
y_ref = ref(x)

print(y_my.shape, y_ref.shape)          # both (2, 3, 4)
print(torch.allclose(y_my, y_ref, 1e-6))


torch.Size([2, 3, 4]) torch.Size([2, 3, 4])
False


In [22]:
class RMSNorm(nn.Module):
    def __init__(self, normalized_shape: int, eps: float = 1e-8):
        super().__init__()
        self.normalized_shape = normalized_shape
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(normalized_shape))



    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        RMSNorm: x / sqrt(mean(x^2) + eps) * weight
        over the last dimension.
        """
        mean = torch.mean(x ** 2, dim=-1, keepdim=True) 
        x_hat = (x)/torch.sqrt(mean +self.eps)

        return x_hat * self.weight


In [23]:
torch.manual_seed(0)
x = torch.randn(2, 3, 4)   # (..., D) with D=4

rms = RMSNorm(4)
y = rms(x)

print(x.shape, y.shape)     # both (2, 3, 4)

# Check per-position RMS after norm (ignoring weight)
with torch.no_grad():
    w = torch.ones(4)
    rms.weight.copy_(w)
    y = rms(x)
    rms_val = torch.sqrt((y ** 2).mean(dim=-1))
    print("RMS per position:\n", rms_val)


torch.Size([2, 3, 4]) torch.Size([2, 3, 4])
RMS per position:
 tensor([[1.0000, 1.0000, 1.0000],
        [1.0000, 1.0000, 1.0000]])


## MLPs and residual networks

Now you’ll build larger networks by composing layers.

### MLP
An MLP is a stack of `depth` Linear layers with non-linear activations (use GELU) in between.
In this exercise you’ll support:
- configurable depth
- a hidden dimension
- optional LayerNorm between layers (a common stabilization trick)

A key skill is building networks using `nn.ModuleList` / `nn.Sequential` while keeping shapes consistent.

### Transformer-style FeedForward (FFN)
A transformer block contains a position-wise feedforward network:
- `D -> 4D -> D` (by default)
- activation is typically **GELU**

This is essentially an MLP applied independently at each token position.

### Residual wrapper
Residual connections are the simplest form of “skip connection”:
- output is `x + fn(x)`

They improve gradient flow and allow training deeper networks more reliably.

In [24]:
class MLP(nn.Module):
    def __init__(
        self,
        in_dim: int,
        hidden_dim: int,
        out_dim: int,
        depth: int,
        use_layernorm: bool = False,
    ):
        super().__init__()
        self.in_dim = in_dim
        self.hidden_dim = hidden_dim
        self.out_dim = out_dim
        self.depth = depth
        self.use_layernorm = use_layernorm
        
        layers = []
        if self.use_layernorm:
            layers.extend([
            nn.Linear(self.in_dim,self.hidden_dim),
            nn.GELU(),
            nn.LayerNorm(self.hidden_dim)
            ])
            for _ in range(self.depth-2):
                layers.extend([
                    nn.Linear(self.hidden_dim,self.hidden_dim),
                    nn.GELU(),
                    nn.LayerNorm(self.hidden_dim),
                ])
        else:
            layers.extend([
            nn.Linear(self.in_dim,self.hidden_dim),
            nn.GELU(),
            ])
            for _ in range(self.depth-2):
                layers.extend([
                    nn.Linear(self.hidden_dim,self.hidden_dim),
                    nn.GELU(),
                ])
        layers.extend([
            nn.Linear(self.hidden_dim,self.out_dim),
        ])

        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)




In [25]:
mlp_ln = MLP(in_dim=16, hidden_dim=32, out_dim=8, depth=3, use_layernorm=True)
mlp_no = MLP(in_dim=16, hidden_dim=32, out_dim=8, depth=3, use_layernorm=False)

x = torch.randn(4, 16)

print(mlp_ln(x).shape)   # expect (4, 8)
print(mlp_no(x).shape)   # expect (4, 8)

print(mlp_ln)            # check LayerNorm appears
print(mlp_no)            # check no LayerNorm modules


torch.Size([4, 8])
torch.Size([4, 8])
MLP(
  (net): Sequential(
    (0): Linear(in_features=16, out_features=32, bias=True)
    (1): GELU(approximate='none')
    (2): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
    (3): Linear(in_features=32, out_features=32, bias=True)
    (4): GELU(approximate='none')
    (5): LayerNorm((32,), eps=1e-05, elementwise_affine=True, bias=True)
    (6): Linear(in_features=32, out_features=8, bias=True)
  )
)
MLP(
  (net): Sequential(
    (0): Linear(in_features=16, out_features=32, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): GELU(approximate='none')
    (4): Linear(in_features=32, out_features=8, bias=True)
  )
)


In [26]:
class FeedForward(nn.Module):
    """
    Transformer-style FFN: D -> 4D -> D (default)
    """

    def __init__(self, d_model: int, d_ff: int | None = None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.activation = nn.GELU()
        self.fc2 = nn.Linear(d_ff, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = self.fc1(x)
        x = self.activation(x)
        x = self.fc2(x)

        return x

In [27]:
ff = FeedForward(d_model=64)
x = torch.randn(2, 10, 64)   # (batch, seq, d_model)
y = ff(x)
print(y.shape)               # should be (2, 10, 64)


torch.Size([2, 10, 64])


In [28]:
class Residual(nn.Module):
    def __init__(self, fn: nn.Module):
        super().__init__()
        self.fn = fn

    def forward(self, x: torch.Tensor, *args, **kwargs) -> torch.Tensor:
        # TODO: return x + fn(x, ...)
        return x + self.fn(x, *args, **kwargs)

## Classification problem

In this section you’ll put everything together in a minimal MNIST classification experiment.

You will:
1) download and load the MNIST dataset
2) implement cross-entropy from logits (stable, using log-softmax)
3) build a simple MLP-based classifier (flatten MNIST images first)
4) write a minimal training loop
5) report train loss curve and final accuracy

The goal here is not to reach state-of-the-art accuracy, but to understand the full pipeline:
data → model → logits → loss → gradients → parameter update.

### Model notes
- We want you to combine the MLP we implemented above with the classification head we define below into one model 

### MNIST notes
- MNIST images are `28×28` grayscale.
- After `ToTensor()`, each image has shape `(1, 28, 28)` and values in `[0, 1]`.
- For an MLP classifier, we flatten to a vector of length `784`.

## Deliverables
- Include a plot of your train loss curve in the video submission as well as a final accuracy. 
- **NOTE** Here we don't grade on model performance but we expect you to achieve at least 70% accuracy to confirm a correct model implementation.

In [29]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

In [ ]:
transform = transforms.ToTensor()  # -> float32 in [0,1], shape (1, 28, 28)

train_ds = datasets.MNIST(root="data", train=True, download=True, transform=transform)
test_ds  = datasets.MNIST(root="data", train=False, download=True, transform=transform)

train_dl = DataLoader(
    train_ds,
    batch_size=16,
    shuffle=True,
    num_workers=4
)

test_dl = DataLoader(
    test_ds,
    batch_size=16,
    shuffle=True,
    num_workers=4
)


In [ ]:
def cross_entropy_from_logits(
    logits: torch.Tensor,
    targets: torch.Tensor,
) -> torch.Tensor:
    """
    Compute mean cross-entropy loss from logits.

    logits: (B, C)
    targets: (B,) int64

    Requirements:
    - Use log-softmax for stability (do not use torch.nn.CrossEntropyLoss, we check this in the autograder).
    """
    log_probs = torch.nn.functional.log_softmax(logits, dim=1)
    batch_indices = torch.arange(logits.size(0))
    correct_log_probs = log_probs[batch_indices, targets]
    loss = -correct_log_probs.mean()
    return loss
    

In [ ]:
class ClassificationHead(nn.Module):
    def __init__(self, d_in: int, num_classes: int):
        super().__init__()
        self.cf_head = nn.Linear(d_in, num_classes)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        x: (..., d_in)
        return: (..., num_classes) logits
        """
        logits = self.cf_head(x)
        return logits

In [ ]:
def accuracy(loader):
    # TODO: You can use this function to evaluate your model accuracy.
    raise NotImplementedError

In [ ]:
def train_classifier(
    model: nn.Module,
    train_data_loader: DataLoader,
    test_data_loader: DataLoader,
    lr: float,
    epochs: int,
    seed: int = 0,
) -> list[float]:
    """
    Minimal training loop for MNIST classification.

    Steps:
    - define optimizer
    - for each epoch:
        - sample minibatches
        - forward -> cross-entropy -> backward -> optimizer step
      - compute test accuracy at the end of each epoch
    - return list of training losses (one per update step)

    Requirements:
    - call model.train() during training and model.eval() during evaluation
    - do not use torch.nn.CrossEntropyLoss (use your cross_entropy_from_logits)
    """
    
